# 01 · Sessions and progression funnel

**Question.** How do players play, and where do they stop in the core progression?
**Answer.** The median session lasts about 9.5 minutes. Players rarely stop before the first zone; the only large drop is the first purchase (1.1% of players, Android only).
**Method.** A session is a `user_session_id` (table `sessions`, `sql/01_base_tables.sql`). The funnel is computed on every player seen in the window (`game_events.metrics.funnel`); notebook 02 restricts it to new players.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from game_events import config, metrics
from game_events.data import connect_events
from game_events.queries import build_base_tables

con, readable, unreadable = connect_events()
build_base_tables(con)
print(f"{len(readable)} readable files, {len(unreadable)} skipped")

## 1. Events without a session id

In [ ]:
con.sql("""
    SELECT event_name, COUNT(*) AS events_without_session
    FROM events
    WHERE user_session_id = ''
    GROUP BY 1 ORDER BY 2 DESC
""").show(max_width=200)

con.sql("""
    SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE user_session_id = '') / COUNT(*), 1) AS pct_events_without_session
    FROM events
""").show()

All of them are server-side events (achievements, currencies, items, identity, experiment join): they are left out of session metrics.

## 2. Sessions

In [ ]:
con.sql("SELECT COUNT(*) AS sessions, COUNT(DISTINCT user_session_id) AS distinct_ids FROM sessions").show()

One row per id: no session id is shared between players.

## 3. Session duration

In [ ]:
con.sql("""
    SELECT COUNT(*)                                   AS sessions,
           QUANTILE_CONT(duration_s, 0.5)             AS median_s,
           QUANTILE_CONT(duration_s, 0.9)             AS p90_s,
           QUANTILE_CONT(duration_s, 0.99)            AS p99_s,
           ROUND(AVG(duration_s), 0)                  AS mean_s,
           COUNT(*) FILTER (WHERE duration_s = 0)     AS zero_duration,
           COUNT(*) FILTER (WHERE duration_s > 86400) AS longer_than_24h
    FROM sessions
""").show(max_width=200)

In [ ]:
minutes = con.execute("SELECT LEAST(duration_s, 3600) / 60.0 AS minutes FROM sessions").df()["minutes"]
ax = minutes.plot.hist(bins=60, figsize=(8, 3.2), color="#2a6f97")
ax.set_xlabel("Session duration (minutes, capped at 60)")
ax.set_ylabel("Sessions")
plt.tight_layout()
plt.show()

## 4. Daily active players and sessions per player

In [ ]:
daily = con.execute("""
    SELECT CAST(started_at AS DATE)                           AS day,
           COUNT(DISTINCT user_id)                            AS players,
           ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT user_id), 2) AS sessions_per_player
    FROM sessions
    GROUP BY 1 ORDER BY 1
""").df()

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(daily["day"], daily["players"], color="#2a6f97")
for day, label in [(config.PARTIAL_DAYS[0], "partial day"), (config.GAP_DAY, "data gap")]:
    row = daily[daily["day"].astype(str) == day]
    if len(row):
        ax.annotate(label, (row["day"].iloc[0], row["players"].iloc[0]),
                    xytext=(0, -18), textcoords="offset points", ha="center", fontsize=8)
ax.set_ylabel("Players with a session")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
daily.head()

## 5. Progression funnel, all players

Steps: `gameOpened` → `tokenCashout` → `deviceUnlocked` → `zoneUnlocked` → `purchaseCompleted`. A step counts only if it happens after the previous one. The steps are inferred from the event names (`docs/tracking_plan.md`), not from game design documents.

In [ ]:
funnel = metrics.funnel(con, population="all")
funnel[["event", "players", "pct_of_previous", "pct_of_first"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(funnel["event"][::-1], funnel["players"][::-1], color="#2a6f97")
for i, (n, pct) in enumerate(zip(funnel["players"][::-1], funnel["pct_of_first"][::-1], strict=True)):
    ax.text(n, i, f"  {n:,}  ({pct}%)", va="center", fontsize=8)
ax.set_xlabel("Players")
ax.set_xlim(0, funnel["players"].max() * 1.25)
plt.tight_layout()
plt.show()

## Findings

- **Events without a session id.** 10.4% of events (mostly `achievementUpdated`, `currencyGranted`, `currencySpent`) have an empty `user_session_id` and are excluded from sessions. Session figures describe the other 89.6%.
- **Session length.** 275,697 sessions. Median 568 s (about 9.5 minutes), 90th percentile 3,891 s (about 65 minutes), mean 2,186 s (about 36 minutes). The mean is almost four times the median: use the median. 12,601 sessions (4.6%) have a duration of 0, and 437 (0.2%) last more than 24 hours. A duration is the gap between the first and last event of a session, so time spent with the app in the background probably counts too (not verified).
- **Activity over time.** Daily active players fall from about 4,900 in the first week of March to about 3,200 on 12 March, then oscillate between 3,400 and 4,600. 2025-03-27 (about 1,400 players) is an unexplained drop; 2025-02-28 is a partial day. Players have roughly 1.5 to 1.8 sessions per day on the first days shown.
- **Funnel (all players in the window).** 68,881 players opened the game; 91.8% reached `tokenCashout`, 81.0% `deviceUnlocked`, 75.7% `zoneUnlocked`, and 1.1% (777 players) `purchaseCompleted`. Losses before the purchase are small; the purchase is the only large drop. Caution: `purchaseCompleted` is only recorded for Android players (iPhone purchases are not visible there, see `docs/tracking_plan.md`), so the last step understates purchases.
- **Limits.** The funnel steps are an interpretation of event names, not documented game design. The funnel includes returning players, whose earlier progress happened before the window, so steps 3 and 4 are underestimated for them (a new-player funnel is built in notebook 02).